# Fetch Guild List (How-to-use)

1. **Set Parameters:**  
   Use the widgets to select `pages`, `limit`, and `debug` mode.

2. **Run Notebook:**  
   Just hit "Run All" after setting parameters. The following happens:
   - Imports dependencies and settings.
   - Fetches and combines guild data across the selected pages.
   - Saves results as JSON.
   - Output is printed and saved for analysis.

3. **Debug Mode:**  
   If `debug` is `True`, uses a sample payload.

## Import Dependencies, Job Settings, Schemas, and Helpers

1. Imports all required Python libraries
2. Add and import custom job settings, schema definitions, and helper functions.

In [0]:
import asyncio
import json
from datetime import datetime
from httpx import AsyncClient
from typing import ClassVar

import pandas as pd
import numpy as np
from pydantic import (
    BaseModel, 
    field_validator, 
    Field,
    model_serializer
)
from unittest.mock import patch, AsyncMock, MagicMock

In [0]:
from schemas import GraphQLQuery
from settings import fflogs_settings as settings

## Seting this notebook's parameters

In [0]:
def define_task_widget():
    dbutils.widgets.dropdown(
        name="limit",
        defaultValue="1",
        choices=["1", "5", "10"],
        label="Number of pages to fetch"
    )
    dbutils.widgets.dropdown(
        name="per_page_records",
        defaultValue="100",
        choices=["50", "100", "200"],
        label="Number of records per page"
    )
    dbutils.widgets.dropdown(
        name="debug",
        defaultValue="True",
        choices=["True", "False"],
        label="DEBUG mode runs test function"
    )
    dbutils.widgets.text(
        name="offset",
        defaultValue="0",
        label="Page to start pulling from"
    )

## Guide Query & Payload

1. Presents the GraphQL query used to fetch guilds.
2. A sample of the expected successful response payload (also usable later in `debug` mode to test flow)

In [0]:
class GuildListQuery(GraphQLQuery):
    QUERY: ClassVar[str] = """
    query GetGuilds(
      $page: Int!,
      $limit: Int!,
      $zoneId: Int
    ) {
      guildData {
        guilds(
          page: $page,
          limit: $limit
        ) {
          total
          has_more_pages
          
          data {
            id
            name
            description
            type

            server {
              id
              region { id }
            }

            zoneRanking(zoneId: $zoneId) {
              progress {
                worldRank { number }
                regionRank { number }
              }

              speed {
                worldRank { number }
                regionRank { number }
              }
            }
          }
        }
      }
    }
    """
    page: int = 1
    limit: int = 100
    zone_id: int | None = None

In [0]:
GUILD_LIST_JSON = {
    "guildData": {
        "guilds": {
            "total": 250,
            "has_more_pages": True,
            "data": [
                {
                    "id": 12345,
                    "name": "Example Guild",
                    "description": "A sample guild for testing",
                    "type": 0,
                    "server": {
                        "id": 34,
                        "region": {"id": 1}
                    },
                    "zoneRanking": {
                        "progress": {
                            "worldRank": {"number": 150},
                            "regionRank": {"number": 45}
                        },
                        "speed": {
                            "worldRank": {"number": 200},
                            "regionRank": {"number": 60}
                        }
                    }
                },
                {
                    "id": 67890,
                    "name": "Another Guild",
                    "description": "Second sample guild",
                    "type": 0,
                    "server": {
                        "id": 35,
                        "region": {"id": 1}
                    },
                    "zoneRanking": None
                }
            ]
        }
    }
}

## Send Query to Endpoint & Batch Results

1. Send a GraphQL query to the endpoint asynchronously
2. Batch multiple queries with `asyncio.gather` concurrency to combine multiple pages.
3. Write results to volume

In [0]:
async def query_graphql(
    client: AsyncClient,
    query: GraphQLQuery, 
    access_token: str
) -> dict:
    """Executes a GraphQL query against the configured endpoint asynchronously.

    Args:
        client (AsyncClient): The httpx async client to use for the request.
        query (GraphQLQuery): The GraphQL query object to execute.
        access_token (str): Bearer token used to authenticate the request.

    Returns:
        dict: The 'data' field of the GraphQL response.

    Raises:
        ValueError: If the GraphQL response contains errors.
        httpx.HTTPStatusError: If the HTTP request fails.
    """
    _FFLOGS_QUERY_ENDPOINT = "https://www.fflogs.com/api/v2/client"

    response = await client.post(
        _FFLOGS_QUERY_ENDPOINT,
        json=query.model_dump(),
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
        },
    )
    response.raise_for_status()
    
    return response.json()['data']

In [0]:
async def batch_query_graphql(
    client: AsyncClient,
    access_token: str,
    offset: int = 0,
    limit: int = 10,
    per_page_records: int = 100,
) -> dict:
    """Fetches and aggregates guild data across multiple pages asynchronously.

    Args:
        client (AsyncClient): The httpx async client to use for requests.
        access_token (str): Bearer token used to authenticate the request.
        offset (int, optional): Page offset to start fetching from. Defaults to 0.
        limit (int, optional): Number of pages to fetch. Defaults to 10.
        per_page_records (int, optional): Number of results per page. Defaults to 100.

    Returns:
        dict: The aggregated GraphQL response with all guilds combined.

    Raises:
        ValueError: If any GraphQL response contains errors.
        httpx.HTTPStatusError: If any HTTP request fails.
    """
    queries = [
        GuildListQuery(page=page, limit=per_page_records)
        for page in range(offset + 1, offset + limit + 1)
    ]
    results = await asyncio.gather(*[
        query_graphql(client, query, access_token)
        for query in queries
    ])

    all_guilds = []
    for data in results:
        all_guilds.extend(
            data["guildData"]["guilds"]["data"]
        )

    results[0]["guildData"]["guilds"]["data"] = all_guilds

    return results[0]

In [0]:
def write_json_to_volume(
    data: dict, 
    volume_dir: str,
    file_name: str
) -> None:
    now = datetime.now().strftime("%Y%m%d_%H%M%S")

    with open(f"{volume_dir}/{file_name}_{now}.json", "w") as f:
        json.dump(data, f, indent=4)

## Orchestrate All Tasks & Debug/Test Mode

1. Coordinates the entire workflow
2. Running test functions when debug mode is enabled.

In [0]:
async def query_guilds_to_volume() -> None:
    define_task_widget()

    offset = int(dbutils.widgets.get("offset"))
    limit = int(dbutils.widgets.get("limit"))
    per_page_records = int(dbutils.widgets.get("per_page_records"))

    async with AsyncClient() as client:
        result = await batch_query_graphql(
            client=client,
            access_token=settings.fflogs_access_token,
            offset=offset,
            limit=limit,
            per_page_records=per_page_records,
        )
    
    write_json_to_volume(
        data=result,
        volume_dir=settings.guilds_volume,
        file_name="guilds"
    )
    
    print(f"New batch of guilds have been written to volume {settings.guilds_volume}")

In [0]:
async def test_query_guilds_to_volume() -> None:
    mock_client = AsyncMock(spec=AsyncClient)

    with (
        patch("httpx.AsyncClient.__aenter__", return_value=mock_client),
        patch("httpx.AsyncClient.__aexit__", return_value=False),
        patch("__main__.batch_query_graphql",
            new_callable=AsyncMock, return_value=GUILD_LIST_JSON) as mock_batch,
        patch("__main__.write_json_to_volume") as mock_write,
    ):
        await query_guilds_to_volume()

        mock_batch.assert_awaited_once_with(
            client=mock_client,
            access_token=settings.fflogs_access_token,
            offset=int(dbutils.widgets.get("offset")),
            limit=int(dbutils.widgets.get("limit")),
            per_page_records=int(dbutils.widgets.get("per_page_records")),
        )
        mock_write.assert_called_once_with(
            data=GUILD_LIST_JSON,
            volume_dir=settings.guilds_volume,
            file_name="guilds"
        )

    print(json.dumps(GUILD_LIST_JSON, indent=4))
    print("✓ test_query_guilds_to_volume passed")

In [0]:
if dbutils.widgets.get("debug") == "True":
    await test_query_guilds_to_volume()
else:
    await query_guilds_to_volume()